# 🍜 頑固ラーメン屋のAIを作ろう (Build a Stubborn Ramen AI)

---
```text
 FFFFF   A   QQQQ      BBBB   OOO  TTTTT
 F      A A  Q  Q      B  B  O   O   T
 FFF   AAAAA Q  Q      BBBB  O   O   T
 F     A   A Q  Q      B  B  O   O   T
 F     A   A  QQ Q     BBBB   OOO    T
```
# 📙 ラボ2（午後）：AIの「脳」をアップグレードしよう — Transformer
### (Lab 2: Brain Transplant — Building a Transformer from Scratch)
---

午前のモデルは「直前の1文字」しか見ていませんでした（金魚の記憶力）。
これでは「ラーメン」と言いたいのに、「ラ」の次に「ク」が来て「ラクダ」になってしまうかもしれません。

午後は、GoogleやOpenAIが使っている技術 **「Transformer (トランスフォーマー)」** を導入します。

### 変わること (Changes)
1. **Self-Attention (自己アテンション)**: AIが「過去の会話」を振り返るようになります。
2. **Context (文脈)**: 「いらっしゃい」と言われたら「ませ」と返す、といった長い繋がりを理解します。
3. **Complexity (複雑さ)**: コードは長くなりますが、やることは同じ **「次に来る文字の予測」** です。

### 本ラボの構成
- **前半**: Transformerをゼロから構築し、ラーメン屋データで訓練する
- **後半**: 🌟 **あなた自身のFAQデータ** でAIを訓練する（持ち帰りSLMの第一歩！）


# ==============================================================================
# ✅ ステップ0：準備（ライブラリとデータの再読み込み）
# ==============================================================================
このノートブックは午前とは別ファイルなので、ライブラリとデータをもう一度読み込みます。
（内容は午前のパート0〜1と同じです。実行するだけでOK）


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import japanize_matplotlib

torch.manual_seed(1337)

# --- 午前と同じラーメン屋データ ---
text = """
客：いらっしゃい。
店：へい、いらっしゃい。食券買ってな。
客：おすすめは何ですか？
店：うちは塩ラーメンしか置いてないよ。メニューをよく見なさい。
客：すいません、お水ください。
店：水はセルフサービスだよ。あそこの給水機を使ってくれ。
客：大盛りはできますか？
店：うちは大盛りやってないんだ。味のバランスが崩れるからね。
客：麺の硬さは選べますか？
店：うちは「普通」が一番うまいんだ。黙って座って待ってな。
客：ごちそうさまでした。
店：おう、まいど。丼はカウンターに上げてってな。
客：トイレはどこですか？
店：店の外を出て右だよ。
客：替え玉お願いします。
店：だから、メニュー見てくれよ。替え玉もやってないんだ。
""" * 100

print(f"データの文字数: {len(text)} 文字")

## 前処理を「関数」にまとめる (Refactoring into Functions)

午前は前処理を1行ずつ書きましたが、午後の後半で **別のデータ（あなたのFAQ）** でも
同じ処理を使い回します。そこで、前処理一式を関数 `build_dataset` にまとめておきます。

> 💡 **プログラミングの知恵**: 同じ処理を2回以上使うなら、関数にまとめる。
> これで「データを差し替えて再訓練」がたった1行でできるようになります。


In [ ]:
def build_dataset(raw_text):
    """テキストを受け取り、語彙・変換関数・訓練/検証テンソルを一括で作って返す"""
    chars = sorted(list(set(raw_text)))            # ユニーク文字のリスト（語彙）
    vocab_size = len(chars)
    stoi = { ch:i for i,ch in enumerate(chars) }   # 文字 → 番号
    itos = { i:ch for i,ch in enumerate(chars) }   # 番号 → 文字
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda l: ''.join([itos[i] for i in l])

    data = torch.tensor(encode(raw_text), dtype=torch.long)  # 一度だけテンソル化
    n = int(0.9 * len(data))                                  # 90% / 10% に分割
    train_data, val_data = data[:n], data[n:]
    return vocab_size, encode, decode, train_data, val_data

# ラーメン屋データで実行
vocab_size, encode, decode, train_data, val_data = build_dataset(text)
print(f"語彙数: {vocab_size} / 訓練: {len(train_data)} トークン / 検証: {len(val_data)} トークン")

# ==============================================================================
# ✅ ステップ1：設定の更新 (Hyperparameters)
# ==============================================================================

モデルの「設計図の数値」＝ **ハイパーパラメータ** を決めます。
これらは学習前に人間が設定する値で、モデルのサイズ・賢さ・計算コストを左右します。

| 名前 | 意味 | 例え |
| :--- | :--- | :--- |
| `block_size` | 記憶力（一度に見る文字数） | 金魚(1文字) → **人間(32文字)** へ！ |
| `n_embd` | 各文字を表すベクトルの次元数 | 1文字に持たせる「情報の豊かさ」 |
| `n_head` | アテンションヘッドの数 | 何個の視点から同時に文脈を見るか |
| `n_layer` | Transformerブロックの層数 | 思考の深さ |
| `dropout` | 学習中にランダムに一部を無視する率 | 丸暗記(過学習)を防ぐ工夫 |


In [ ]:
# 脳のスペックを上げます
batch_size = 32   # 一度に処理する量
block_size = 32   # 記憶力 (午前は1文字でしたが、午後は32文字まで覚えます！)
max_iters = 3000  # 学習回数
eval_interval = 500   # 何回ごとに成績(Loss)を測るか
learning_rate = 1e-3
n_embd = 64       # ベクトルの次元数
n_head = 4        # ヘッド数
n_layer = 4       # レイヤー数
dropout = 0.0     # ドロップアウト率（今日はデータが単純なので0）

# GPUがあれば使う（なければCPU）
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用デバイス: {device}")

In [ ]:
# ------------------------------------------------------------------------------
# データの読み込み方も、長い記憶力 (block_size=32) に合わせて更新します
# ------------------------------------------------------------------------------
def get_batch(split, train_data, val_data):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, train_data, val_data, eval_iters=100):
    """訓練/検証の両方でLossを測定する（午前と同じ関数）"""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, train_data, val_data)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

print(f"設定完了！記憶力が {block_size} 文字にアップグレードされました。")

# ==============================================================================
# ✅ ステップ2：Transformerの部品を組み立てる
# ==============================================================================

ここがワークショップの心臓部です。少し複雑に見えますが、部品は4つだけです。

```text
 ① Head (アテンションヘッド)      : 過去のどの文字に注目すべきかを計算
 ② MultiHeadAttention             : ①を複数の視点で同時に実行
 ③ FeedForward                    : 集めた情報を整理して「考える」
 ④ Block                          : ①〜③をひとまとめにした基本単位（これを積み重ねる）
```

## 🔑 最重要概念：自己アテンション (Self-Attention) — QKVの図書館アナロジー

自己アテンションは **図書館で本を探すプロセス** に例えられます：

| 記号 | 名前 | 図書館での例え | モデル内での意味 |
| :--- | :--- | :--- | :--- |
| **Q** | クエリ (Query) | あなたの質問「AIの本は？」 | 今注目している文字が「探している情報」 |
| **K** | キー (Key) | 各本の背表紙のラベル | 各文字が「自分はこういう情報を持っている」という看板 |
| **V** | バリュー (Value) | 本の中身そのもの | 各文字が実際に持っている情報 |

自分のQと全文字のKを照合して関連度を計算し、関連度が高い文字のVを重み付けして集めます。

例：「その猫はマットの上に座っていた。**それ**はふわふわだった」
→ 「それ」のクエリは「猫」のキーと強く反応し、「猫」の情報を取り込みます。
これが **文脈を理解する** 仕組みの正体です。

## 🎭 因果マスク (Causal Mask) — カンニング防止
言語モデルは「次の文字」を当てるゲームです。もし未来の文字が見えたらカンニングになってしまいます。
そこで、**未来方向の関連度を `-inf` で塗りつぶして見えなくする** のが因果マスクです。

**※ 以下の2セルは変更しなくてOKです。コピー＆ペースト感覚で実行だけしてください。**


In [ ]:
class Head(nn.Module):
    """ ① アテンション・ヘッド：過去の重要な情報を探す機能 """
    def __init__(self, head_size):
        super().__init__()
        # 入力ベクトルから Q, K, V それぞれを作るための変換（線形層）
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # 下三角行列（未来を隠すマスクの材料）。学習パラメータではないので register_buffer
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # 各文字の「看板」
        q = self.query(x)  # 各文字の「質問」

        # ここで過去の文字との関連性を計算しています (Q・Kの内積)
        # C**-0.5 を掛けるのは、数値が大きくなりすぎて学習が不安定になるのを防ぐため
        wei = q @ k.transpose(-2, -1) * C**-0.5      # (B, T, T) 全文字ペアの関連度表

        # 因果マスク：未来の文字への関連度を -inf にして「見えなく」する
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # softmaxで関連度を「合計1の確率（注目度）」に変換。-inf の場所は確率0になる
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # 注目度に従って、各文字のV（情報の中身）を重み付けして集める
        v = self.value(x)
        out = wei @ v
        return out


class MultiHeadAttention(nn.Module):
    """ ② 複数のヘッドで同時に考える機能 """
    # 1つのヘッドは1つの視点しか持てません。
    # 「文法に注目するヘッド」「意味に注目するヘッド」など、
    # 複数の視点の結果を連結して、より豊かな文脈表現を作ります。
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)  # 連結結果を元の次元に整える層
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # 各ヘッドの出力を連結
        out = self.dropout(self.proj(out))
        return out


class FeedForward(nn.Module):
    """ ③ 考えたことを整理する機能 """
    # アテンションが「情報を集める」役割なら、こちらは集めた情報で「考える」役割。
    # 一度4倍の広さに展開して(ReLUで)考え、元のサイズに戻すシンプルな構造です。
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ ④ これらをひとまとめにしたブロック（Transformerの基本単位） """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa   = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1  = nn.LayerNorm(n_embd)   # 層正規化：データ分布を整える「安定化装置」
        self.ln2  = nn.LayerNorm(n_embd)

    def forward(self, x):
        # 「x = x + ...」 が残差接続 (Residual Connection)。
        # 元の情報を失わずに、新しい情報だけを「付け加える」イメージ。
        # 層を深く重ねても学習が壊れない（勾配が消えない）ための重要な発明です。
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

print("Transformerの部品①〜④が完成しました！")

## 部品を組み立てて、完全なGPTモデルにする

最後に部品を積み重ねます。データの流れは以下の通りです：

```text
 入力トークン列 「客：おすすめは」
      │
      ▼
 ① トークン埋め込み ─┬─▶ ＋ ──▶ ② Transformerブロック × 4層 ──▶ ③ 最終正規化
 ① 位置埋め込み ─────┘                                              │
                                                                     ▼
                                            ④ lm_head: 各文字のスコア(logits)を出力
                                               「何」: 高スコア  「水」: 低スコア ...
```

- **トークン埋め込み**: 文字ID（ただの番号）を、意味を持つベクトルに変換
- **位置埋め込み**: アテンションは語順を知らないため、「何番目の文字か」という情報を足す
- **lm_head**: 最終ベクトルを「語彙の全文字に対するスコア」に変換。最高スコアの文字が予測


In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 文字をベクトルに変換（意味を持たせる）
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        # 「文字の位置」を覚える機能 (Position Embedding)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # トランスフォーマーのブロックを積み重ねる
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)  # スコア(logits)への変換

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                             # 文字の意味
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # 文字の位置
        x = tok_emb + pos_emb        # 意味 + 位置 を足し合わせる
        x = self.blocks(x)           # 4層のブロックで文脈を読み込む
        x = self.ln_f(x)
        logits = self.lm_head(x)     # 次の文字のスコアを出力

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # 過去 block_size 分の文字だけを見るように切り取る
            # （position_embedding が block_size までしか対応していないため）
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]                          # 最後の位置の予測を使う
            probs = F.softmax(logits, dim=-1)                  # スコア → 確率
            idx_next = torch.multinomial(probs, num_samples=1) # 確率に従って抽選
            idx = torch.cat((idx, idx_next), dim=1)            # 文脈に追加して繰り返す
        return idx

print("GPTモデル（トランスフォーマー）の定義完了！")

# ==============================================================================
# ✅ ステップ3：新しい脳でトレーニング
# ==============================================================================

学習ループは **午前とまったく同じ4ステップ** です。脳が変わっても、勉強法は同じ！

後半で再利用するため、訓練処理を関数 `train_model` にまとめます。


In [ ]:
def train_model(model, train_data, val_data, max_iters=3000):
    """モデルを訓練し、Lossの履歴を返す（午前と同じ4ステップのループ）"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    history = {"iter": [], "train": [], "val": []}

    for it in range(max_iters):
        xb, yb = get_batch('train', train_data, val_data)

        logits, loss = model(xb, yb)                  # 1. 予測と誤差計算
        optimizer.zero_grad(set_to_none=True)         # 2. 前回のゴミを掃除
        loss.backward()                               # 3. 逆伝播（原因を探る）
        optimizer.step()                              # 4. パラメータ更新

        if it % eval_interval == 0 or it == max_iters - 1:
            losses = estimate_loss(model, train_data, val_data)
            history["iter"].append(it)
            history["train"].append(losses['train'].item())
            history["val"].append(losses['val'].item())
            print(f"回数: {it}, 訓練Loss: {losses['train']:.4f}, 検証Loss: {losses['val']:.4f}")
    return history


# 新しい GPTモデルを初期化して訓練！
model = GPTLanguageModel(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"モデルのパラメータ数: {n_params:,} 個")
print(f"レベルアップした脳で {max_iters} 回の学習を開始します...")
print("ヒント: Lossが 2.0 を切ると、かなり賢くなります。\n")

history = train_model(model, train_data, val_data, max_iters=max_iters)
print("学習完了！")

In [ ]:
# 📉 学習曲線：午前のBigram(2.0前後で限界)と比べてどこまで下がったでしょうか？
plt.figure(figsize=(8, 4))
plt.plot(history["iter"], history["train"], marker="o", label="訓練 Loss (train)")
plt.plot(history["iter"], history["val"],   marker="s", label="検証 Loss (val)")
plt.axhline(y=2.0, color="gray", linestyle="--", label="午前のBigramの限界ライン (約2.0)")
plt.title("Transformerの学習曲線")
plt.xlabel("学習回数 (Iteration)")
plt.ylabel("Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ==============================================================================
# ✅ ステップ4：完成したAIと話そう
# ==============================================================================

チャット関数に、ひとつ実用的な工夫 **「✂️ カットオフ処理」** を入れます。

言語モデルは「続きを書く小説家」なので、店長のセリフの後に **勝手に客のセリフまで**
書き始めてしまいます。そこで「客：」が出てきたらそこで切り落とします。
（ChatGPTなどの実サービスも、同様の「停止処理」をプログラムで行っています）


In [ ]:
import sys

def chat_with_smart_ai(start_text, model, encode, decode):
    try:
        context = torch.tensor([encode(start_text)], dtype=torch.long).to(device)
    except KeyError:
        print("エラー: 学習データにない文字が含まれています。")
        return

    print(f"あなた: {start_text}", end="")
    sys.stdout.flush()

    generated_indices = model.generate(context, max_new_tokens=100)[0].tolist()
    full_text = decode(generated_indices)
    ai_response = full_text[len(start_text):]   # 入力した部分を取り除く

    # =================================================================
    # ✂️ カットオフ処理 (The Cut-off Logic)
    # AIが勝手に「客：」のセリフを書き始めたら、そこで切り落とします
    # =================================================================
    if "客：" in ai_response:
        ai_response = ai_response.split("客：")[0]

    print(f"AI店長(GPT): {ai_response}")
    print("-" * 30)

print("--- 頑固ラーメンGPT ---")
print("終了するには「終了」と入力してください")

while True:
    user_input = input("入力してください (例: 客：おすすめは？) >> ")
    if user_input == "終了" or user_input == "exit":
        break
    chat_with_smart_ai(user_input, model, encode, decode)

<br>
<hr>

# ==============================================================================
# 🌟 ステップ5：あなた自身のデータでAIを訓練しよう！ (Train on YOUR Data)
# ==============================================================================

ここからが本日のハイライトです。ラーメン屋の代わりに、**あなたの会社・製品・業務** の
FAQデータでAIを訓練します。

## 手順1: 生成AIでFAQデータセットを作る

下のプロンプトテンプレートをコピーし、`[角括弧]` の部分を書き換えて、
**Claude や ChatGPT などの生成AI** に貼り付けてください。

---
```text
こんにちは。私は、簡単なFAQチャットボットとして機能する小規模な言語モデルを
トレーニングするためのデータセットを作成しています。

あなたのタスクは、以下のトピックに関するよくある質問とその回答のリストを生成することです：
[ここに、あなたの会社、製品、またはトピックの説明を記述してください。
 例：「『蒸し豆珈琲店』という架空のコーヒーショップについて」
     「『タスクマスターPro』という新しい生産性向上アプリについて」]

[例：30]個の、ユニークな質問と回答のペアを生成してください。

このデータはコンピュータプログラムで使用するため、必ず以下の正確なフォーマットで
出力してください。

守ってほしいフォーマットルール：
Q: [質問文]
A: [回答文]

Q: [別の質問文]
A: [別の回答文]

・すべての質問は、必ず Q: （コロンと半角スペース）で始めてください。
・すべての回答は、必ず A: （コロンと半角スペース）で始めてください。
・各Q&Aペアの間には、必ず1行の空白行を入れてください。
・回答は、比較的短く、要点をまとめてください（1〜2文が理想です）。

それでは、データセットの生成を開始してください。
```
---

## 手順2: `my_faq.txt` として保存する

1. JupyterLabの左サイドバー 📁 で、このノートブックと同じフォルダにいることを確認
2. メニュー **File → New → Text File** で新規テキストファイルを作成
3. 生成AIの出力（Q:/A:のペア）を貼り付け
4. ファイル名を **`my_faq.txt`** に変更して保存（右クリック → Rename）

> 💡 データ量のコツ: 今日の小さなモデルは文字単位で学習するため、
> Q&Aは30個以上あると効果的です。少ない場合は下のセルが自動で繰り返して水増しします。


In [ ]:
# 手順3: あなたのFAQデータを読み込んで、新しいAIを訓練します！
# （my_faq.txt が無い場合は、サンプルとしてラーメン屋データで動きます）
import os

if os.path.exists("my_faq.txt"):
    with open("my_faq.txt", "r", encoding="utf-8") as f:   # utf-8指定で日本語の文字化けを防ぐ
        my_text = f.read()
    print(f"✅ my_faq.txt を読み込みました（{len(my_text)} 文字）")

    # 文字単位のモデルにはデータ量が必要なので、小さいデータは繰り返して水増しします
    repeat = max(1, 30000 // max(len(my_text), 1))
    my_text = my_text * repeat
    print(f"   {repeat} 回繰り返して {len(my_text)} 文字に増量しました")
else:
    print("⚠️ my_faq.txt が見つかりません。サンプル（ラーメン屋）データを使います。")
    my_text = text

# 前処理 → モデル初期化 → 訓練。関数化したおかげで、たったこれだけ！
my_vocab_size, my_encode, my_decode, my_train, my_val = build_dataset(my_text)
print(f"\n語彙数: {my_vocab_size}")

my_model = GPTLanguageModel(my_vocab_size).to(device)
print("あなた専用のAIを訓練します...\n")
my_history = train_model(my_model, my_train, my_val, max_iters=3000)
print("\n🎉 あなた専用のAIが完成しました！")

In [ ]:
# あなた専用のAIとチャット！
# コツ: 学習データと同じ形式「Q: 」で書き始めると、AIが「A: 」と答えやすくなります

def chat_with_my_ai(start_text):
    try:
        context = torch.tensor([my_encode(start_text)], dtype=torch.long).to(device)
    except KeyError:
        print("エラー: 学習データにない文字が含まれています。")
        return

    print(f"あなた: {start_text}")
    generated_indices = my_model.generate(context, max_new_tokens=100)[0].tolist()
    ai_response = my_decode(generated_indices)[len(start_text):]

    # ✂️ カットオフ: 次の「Q: 」を書き始めたら切り落とす
    if "Q: " in ai_response:
        ai_response = ai_response.split("Q: ")[0]

    print(f"AI: {ai_response}")
    print("-" * 30)

print("--- あなた専用FAQボット ---")
print("終了するには「終了」と入力してください")

while True:
    user_input = input("入力してください (例: Q: 営業時間は？) >> ")
    if user_input == "終了" or user_input == "exit":
        break
    chat_with_my_ai(user_input)

<br>
<hr>

# 🎓 ラボ2 完了：ゼロからのSLM構築、達成！

| | 午前：Bigram (金魚) | 午後：Transformer (GPT) |
| :--- | :--- | :--- |
| **記憶力** | 直前の1文字だけ | 過去32文字 (文脈) |
| **Loss (誤差)** | 2.5 付近で限界 | **1.5 以下** に激減！ |
| **会話能力** | 宇宙語・支離滅裂 | 文脈を理解した応答 |
| **データ** | ラーメン屋 | 🌟 **あなたのFAQ** |
| **正体** | 次の文字のサイコロ | **高度な次の文字予測機** |

### 💡 最後に：AIが「勝手に客のセリフ」を喋った理由
チャット実験で、AIが勝手に続きの質問まで書き始めませんでしたか？

これが **LLM (大規模言語モデル)** の本質です。
AIは「質問に答えるロボット」ではなく、**「与えられたテキストの続きをひたすら書く小説家」** なのです。
ChatGPTなどの実サービスは、AIが勝手に続きを書かないように、プログラムで「ストップ！」と
命令しているだけなんですね（今日私たちが書いた ✂️ カットオフ処理と同じ発想です）。

### 🤔 でも、まだ実用には遠い…？
その通りです。文字単位・数十万パラメータのモデルでは、学習データの「真似」が限界です。
実務で使うには：
1. **もっと大きなモデル**（数億パラメータ〜）
2. **もっと大量の事前学習**（インターネット規模のテキスト）

…が必要ですが、それを **ゼロからやる必要はありません**。
世界中の企業が公開している「事前学習済みモデル」を、自分のデータで **微調整（ファインチューニング）**
すればよいのです。

### 🚀 次のラボ：本物のSLMを自分のデータで微調整する
**`lab3_lora_and_mcp.ipynb`** に進みましょう。持ち帰って本当に使えるSLMを作ります！
